# Accuracy/F1 of Our Models on Our Survey Answers

### Choose Models to Test

In [ ]:
import os
os.chdir("../..")

In [ ]:
from App.ModelWrapper.model_wrapper import Model

models = [
    Model('language', 'MixedCNN', model='RoBERTa'),
    Model('language', 'MixedCNN', model='RoBERTaLarge'),
    Model('language', 'Classification', model='RoBERTa'),
    Model('language', 'Classification', model='RoBERTaLarge')
]

### Testing Accuracy for Each Trait

In [ ]:
from evaluate import load
import numpy as np
import pandas as pd
from tqdm import tqdm

accuracy = load("accuracy")
f1_metric = load("f1")

batch_size = 64

for model in models:
    trait = model.name.split(" ")[0]
    df = pd.read_parquet(f"Training/DATA/{trait}/{'short_' if trait == 'political' else ''}test.parquet")
    if trait == 'political':
        trait = 'political_view'
    if trait == 'mbti':
        trait = 'type'
    invert_labels = {v: k for k, v in model.label_map.items()} if model.label_map != None else lambda x: x

    all_preds = []
    all_refs = df[trait].map(invert_labels).to_numpy()

    for start in tqdm(range(0, len(df), batch_size), desc=model.name):
        batch = df['text'].iloc[start:start + batch_size].tolist()
        outputs = model.predict(batch)

        match model.type:
            case 'Classification' | 'MixedCNN' | 'CNN':
                outputs = np.argmax(outputs, axis=1)
            case 'Regression':
                outputs = np.round(outputs)

        all_preds.extend(outputs)

    acc = accuracy.compute(predictions=all_preds, references=all_refs)
    f1 = f1_metric.compute(predictions=all_preds, references=all_refs, average="weighted")

    print(f"\n[{model.name}]: \n - accuracy: {acc["accuracy"]:.4f} \n - F1: {f1["f1"]:.4f}\n")
